In [6]:
!nvidia-smi

Mon Mar 16 10:29:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/vizahat36/ai-data-restoration-pipeline.git

fatal: destination path 'ai-data-restoration-pipeline' already exists and is not an empty directory.


In [3]:
%cd ai-data-restoration-pipeline/Part-B

/content/ai-data-restoration-pipeline/Part-B


In [ ]:
!pip install -q \
transformers==4.41.2 \
datasets==2.19.1 \
peft==0.11.1 \
trl==0.9.4 \
accelerate==0.30.1 \
bitsandbytes

In [5]:
!pip install wandb

In [7]:
import os

print(os.listdir("../Part-A"))

['rejected_conversations.jsonl', 'raw_conversations.jsonl', 'generate_dataset.py', 'clean_data.py', 'writeup.md', '.gitkeep', 'quality_report.py', 'cleaned_conversations.jsonl']


Verify Library **Versions**

In [8]:
import transformers, peft, accelerate, datasets

print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("datasets:", datasets.__version__)

transformers: 4.41.2
peft: 0.11.1
accelerate: 0.30.1
datasets: 2.19.1


**dataset** **loading**

In [9]:
import json

dataset = []

with open("../Part-A/cleaned_conversations.jsonl", "r") as f:
    for line in f:
        dataset.append(json.loads(line))

print("Total samples:", len(dataset))
print(dataset[0])

Total samples: 48
{'conversation_id': 'conv_002', 'language': 'hinglish', 'turns': [{'role': 'agent', 'text': 'Hello sir, aapki EMI due hai aur aaj last date hai.'}, {'role': 'customer', 'text': 'Haan, mujhe yaad hai. Main office se nikalte hi payment kar dunga.'}, {'role': 'agent', 'text': 'Great, payment ho jaaye to receipt save kar lijiye.'}, {'role': 'customer', 'text': 'Sure, main UPI se kar deta hoon.'}], 'metadata': {'call_duration_seconds': 391, 'outcome': 'payment_committed'}}


Convert conversations → training samples

In [10]:
formatted_data = []

for item in dataset:
    turns = item["turns"]

    for i in range(len(turns) - 1):

        if turns[i]["role"] == "customer" and turns[i+1]["role"] == "agent":

            user_msg = turns[i]["text"]
            agent_msg = turns[i+1]["text"]

            formatted_data.append({
                "messages": [
                    {"role": "system", "content": "You are a polite EMI collection agent speaking Hinglish."},
                    {"role": "user", "content": user_msg},
                    {"role": "assistant", "content": agent_msg}
                ]
            })

print("Total training samples:", len(formatted_data))
print(formatted_data[0])

Total training samples: 55
{'messages': [{'role': 'system', 'content': 'You are a polite EMI collection agent speaking Hinglish.'}, {'role': 'user', 'content': 'Haan, mujhe yaad hai. Main office se nikalte hi payment kar dunga.'}, {'role': 'assistant', 'content': 'Great, payment ho jaaye to receipt save kar lijiye.'}]}


**Convert to HuggingFace Dataset**

In [22]:
from datasets import Dataset

hf_dataset = Dataset.from_list(formatted_data)

print(hf_dataset)

Dataset({
    features: ['messages'],
    num_rows: 55
})


In [23]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Tokenizer loaded


**Convert messages → training text**

In [ ]:
def format_chat(example):

    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

In [25]:
hf_dataset = hf_dataset.map(format_chat)
hf_dataset = hf_dataset.remove_columns(["messages"])

print(hf_dataset[0])

Map:   0%|          | 0/55 [00:00<?, ? examples/s]

{'text': '<|im_start|>system\nYou are a polite EMI collection agent speaking Hinglish.<|im_end|>\n<|im_start|>user\nHaan, mujhe yaad hai. Main office se nikalte hi payment kar dunga.<|im_end|>\n<|im_start|>assistant\nGreat, payment ho jaaye to receipt save kar lijiye.<|im_end|>\n'}


**Load Model (Important for T4 GPU)**

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

tokenizer.pad_token = tokenizer.eos_token

print("Base model and tokenizer loaded successfully")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Base model and tokenizer loaded successfully


**Configure LoRA**

In [ ]:
from peft import LoraConfig, get_peft_model
import torch 

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(base_model, lora_config)

print("LoRA successfully attached to the model (base_model is now 'model')")

LoRA successfully attached to the model (base_model is now 'model')


**Use SFTTrainer (this fixes your FP16 error)**

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="emi-agent-model",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="no",
    fp16=False, 
    bf16=True
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    dataset_text_field="text",
    tokenizer=tokenizer,
    peft_config=lora_config,
    args=training_args,
    max_seq_length=256
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1965: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:269: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:307: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override

Map:   0%|          | 0/55 [00:00<?, ? examples/s]

In [6]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 16.2 MB/s eta 0:00:00


In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_dataset = hf_dataset.map(tokenize, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

print("Tokenization complete")

Map:   0%|          | 0/55 [00:00<?, ? examples/s]

Tokenization complete


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="emi-agent-model",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="no",
    fp16=True, 
    bf16=False,  
    remove_unused_columns=False,
    report_to="none"

In [31]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [32]:
tokenized_dataset = tokenized_dataset.map(
    lambda x: {"labels": x["input_ids"]}
)

Map:   0%|          | 0/55 [00:00<?, ? examples/s]

In [36]:
def add_labels(example):
    example["labels"] = example["input_ids"].copy()
    return example

tokenized_dataset = tokenized_dataset.map(add_labels)

Map:   0%|          | 0/55 [00:00<?, ? examples/s]

In [33]:
print(tokenized_dataset[0].keys())

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [34]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)
print("Trainer ready")

Trainer ready


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [35]:
trainer.train()

Step,Training Loss
5,6.222500
10,2.084500
15,1.118700
20,0.961800
25,0.819600


TrainOutput(global_step=28, training_loss=2.0751898799623762, metrics={'train_runtime': 5.2358, 'train_samples_per_second': 10.505, 'train_steps_per_second': 5.348, 'total_flos': 30326584442880.0, 'train_loss': 2.0751898799623762, 'epoch': 1.0})

In [36]:
trainer.model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


('lora_adapter/tokenizer_config.json',
 'lora_adapter/special_tokens_map.json',
 'lora_adapter/vocab.json',
 'lora_adapter/merges.txt',
 'lora_adapter/added_tokens.json',
 'lora_adapter/tokenizer.json')

In [37]:
import os
print(os.listdir("lora_adapter"))

['merges.txt', 'tokenizer.json', 'adapter_model.safetensors', 'tokenizer_config.json', 'vocab.json', 'adapter_config.json', 'added_tokens.json', 'README.md', 'special_tokens_map.json']


**Test the Model (Inference)**

In [ ]:
import torch

def generate_response(model, tokenizer, prompt):

    messages = [
        {"role": "system", "content": "You are a polite EMI collection agent speaking Hinglish."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            temperature=0.7,
            do_sample=True
        )

    generated_tokens = outputs[0][inputs.input_ids.shape[-1]:]

    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return response

In [39]:
test_prompts = [
    "Sir main kal EMI payment karunga",
    "Mujhe thoda time chahiye payment ke liye",
    "Next week tak payment kar dunga",
    "Aaj payment nahi kar paunga",
    "Thoda extension mil sakta hai kya"
]

In [40]:
for prompt in test_prompts:
    print("\nCustomer:", prompt)
    print("Agent:", generate_response(model, tokenizer, prompt))


Customer: Sir main kal EMI payment karunga
Agent: Deli ka main paya hain.

Customer: Mujhe thoda time chahiye payment ke liye
Agent: O nehne hain, mai mujhe kuch tak paya ho jaaye hai.

Customer: Next week tak payment kar dunga
Agent: I'm sorry, but I can't assist with that.

Customer: Aaj payment nahi kar paunga
Agent: Keho, hai, kaun se baat hoga!

Customer: Thoda extension mil sakta hai kya
Agent: Sakta hai, but we have to ensure your account is protected.
